# TP Réseaux de neurones artificiels - Rétention clients bancaires

In [ ]:
# J'importe toutes mes librairies
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

## 1. Chargement des données

In [ ]:
df = pd.read_csv('3/Bank_customers.csv')
df.head()

## 2. Exploration des données

In [ ]:
df.shape #Je regarde la taille du dataset pour avoir une idée du volume de données

In [ ]:
df.describe()
# Cette fonction me permet de voir les statistiques descriptives comme la moyenne, l'ecart type, les quartiles etc.

In [ ]:
df.info()
# Je vois les types de données et je repère les colonnes catégorielles (object) et numériques

In [ ]:
df.isnull().sum()
# je regarde si j'ai des données vides pour ne pas bruiter les résultats

In [ ]:
df['Exited'].value_counts()
# Je regarde la distribution de la variable cible pour voir si les classes sont équilibrées
# 0 = le client reste, 1 = le client quitte la banque

## 3. Nettoyage des données - Suppression des colonnes non pertinentes

In [ ]:
# Je supprime RowNumber, CustomerId et Surname car ce sont des identifiants qui n'apportent rien au modèle
# RowNumber = juste un index, CustomerId = identifiant unique, Surname = nom du client
# Ces colonnes n'ont aucun pouvoir prédictif sur le fait qu'un client quitte la banque ou non
df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df.head()

## 4. Visualisation

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True) # J'affiche la matrice de corrélation pour voir les relations entre variables
plt.show()

## 5. Encodage des variables catégorielles

In [ ]:
from sklearn.preprocessing import LabelEncoder

# J'encode Gender en valeur numérique (Male=1, Female=0)
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])

# Pour Geography j'utilise le one-hot encoding avec get_dummies
# drop_first=True pour éviter la multicollinéarité (on garde 2 colonnes au lieu de 3)
df = pd.get_dummies(df, columns=['Geography'], drop_first=True)
df.head()

## 6. Séparation train/test

In [ ]:
from sklearn.model_selection import train_test_split

# Je sépare les features (X) de la variable cible (y)
# Exited est ce qu'on veut prédire : 1 = le client part, 0 = il reste
X = df.drop(columns=['Exited'])
y = df['Exited']

print(X.shape) # Je vérifie le nombre de features, on devrait avoir 11 colonnes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 7. Mise à l'échelle des données

In [ ]:
from sklearn.preprocessing import StandardScaler

# J'utilise StandardScaler pour normaliser les données (moyenne=0, ecart-type=1)
# C'est important pour les réseaux de neurones car les features ont des échelles très différentes
# (ex: Balance en milliers vs HasCrCard en 0/1)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test) # transform seulement sur le test pour éviter le data leakage

## 8. Construction du modèle ANN avec Keras

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Dropout

# Je crée mon modèle séquentiel (les couches s'empilent les unes après les autres)
classifier = Sequential()

# Couche d'entrée + première couche cachée : 6 neurones, activation relu, input_dim=11 car 11 features
classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu', input_dim=11))

# Dropout de 0.1 pour la régularisation (10% des neurones désactivés aléatoirement)
# Ca permet d'éviter le surapprentissage
classifier.add(Dropout(rate=0.1))

# Deuxième couche cachée : 6 neurones aussi
classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu'))
classifier.add(Dropout(rate=0.1))

# Couche de sortie : 1 neurone avec sigmoid car c'est une classification binaire (0 ou 1)
classifier.add(Dense(units=1, kernel_initializer='uniform', activation='sigmoid'))

classifier.summary() # Je regarde l'architecture du réseau

## 9. Compilation du modèle

In [ ]:
# Je compile le modèle avant l'entraînement
# adam = optimiseur qui utilise la descente de gradient stochastique pour ajuster les poids
# binary_crossentropy = fonction de perte adaptée à la classification binaire
# accuracy = métrique pour évaluer la performance
classifier.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

## 10. Entraînement du modèle

In [ ]:
# J'entraîne le modèle avec batch_size=10 et 100 epochs
# batch_size = nombre d'échantillons traités avant mise à jour des poids
# epochs = nombre de fois que le modèle voit l'ensemble des données d'entraînement
classifier.fit(X_train, y_train, batch_size=10, epochs=100)

## 11. Prédictions et évaluation

In [ ]:
# Je fais les prédictions sur le jeu de test
y_pred = classifier.predict(X_test)
y_pred = (y_pred > 0.5) # Je convertis les probabilités en 0 ou 1 avec un seuil de 0.5
print(y_pred[:10]) # J'affiche les 10 premières prédictions pour vérifier

## 12. Matrice de confusion et rapport de classification

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Matrice de confusion pour voir les vrais/faux positifs et négatifs
cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion :")
print(cm)

# Je l'affiche en heatmap pour mieux visualiser
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Prédit')
plt.ylabel('Réel')
plt.title('Matrice de confusion')
plt.show()

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nRapport de classification :")
print(classification_report(y_test, y_pred))
# Le rapport montre la precision, le recall et le f1-score pour chaque classe
# Precision = parmi ceux prédits positifs, combien le sont vraiment
# Recall = parmi les vrais positifs, combien ont été détectés
# F1 = moyenne harmonique des deux

## 13. Prédiction pour un nouveau client

In [ ]:
# Je prédis si ce client va quitter la banque :
# France, CreditScore=600, Homme, Age=40, Tenure=3, Balance=60000, NumOfProducts=2, HasCrCard=1, IsActiveMember=1, Salary=50000

# L'ordre des colonnes doit correspondre à celui du dataframe après encodage :
# CreditScore, Gender, Age, Tenure, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary, Geography_Germany, Geography_Spain
# France => Geography_Germany=0, Geography_Spain=0 (c'est la catégorie de référence)
# Homme => Gender=1

new_client = np.array([[600, 1, 40, 3, 60000, 2, 1, 1, 50000, 0, 0]])
new_client = sc.transform(new_client) # Je normalise avec le même scaler

prediction = classifier.predict(new_client)
print(f"Probabilité de quitter la banque : {prediction[0][0]:.4f}")
print(f"Le client va {'quitter' if prediction[0][0] > 0.5 else 'rester à'} la banque")

## 14. KerasClassifier avec Scikit-Learn et validation croisée k-fold

In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score

# Je crée une fonction qui construit le modèle à chaque fold
# C'est nécessaire pour que KerasClassifier puisse recréer le modèle à chaque itération
def build_classifier():
    classifier = Sequential()
    classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu', input_dim=11))
    classifier.add(Dropout(rate=0.1))
    classifier.add(Dense(units=6, kernel_initializer='uniform', activation='relu'))
    classifier.add(Dropout(rate=0.1))
    classifier.add(Dense(units=1, kernel_initializer='uniform', activation='sigmoid'))
    classifier.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return classifier

# J'utilise KerasClassifier pour intégrer mon modèle Keras dans le workflow scikit-learn
classifier_cv = KerasClassifier(model=build_classifier, batch_size=10, epochs=50, verbose=0)

In [ ]:
# Validation croisée k-fold avec k=10
# Le dataset est découpé en 10 parties, le modèle est entraîné 10 fois
# A chaque fois une partie différente sert de test
# Ca donne une estimation plus fiable de la performance du modèle
accuracies = cross_val_score(estimator=classifier_cv, X=X_train, y=y_train, cv=10)

print(f"Accuracy moyenne : {accuracies.mean():.4f}")
print(f"Ecart-type : {accuracies.std():.4f}")
# Un ecart-type faible signifie que le modèle est stable et ne dépend pas trop du découpage des données